# Conditional Flow Matching.

## Preparing notebook.

### Importing Libraries.

In [1]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from sklearn.model_selection import train_test_split
import torch
from tqdm import tqdm

from labcompass.constants import DataFields, PredictionFields
from labcompass.config import NeuralVelocityFieldConfig
from labcompass.metrics import compute_e_distance, compute_wasserstein_distance, compute_mmd, compute_r_squared
from labcompass.models import FlowMatching
from labcompass.utils import set_reproducibility

sys.path.insert(0, "../../../utils")
from fetch_data import get_log_transformed_experimental_variables


### Constants.

In [2]:
# working locally?
IS_RUN_LOCALLY = False

# paths
if IS_RUN_LOCALLY:
    H5AD_PATH = "/Users/lorenzo.consoli/Work/data/collab-goettgens/ds_HID01_logicle_100k_UMAP_ann.h5ad"
else:
    H5AD_PATH = "../../../../../../project_folder/gdrive/ds_HID01_logicle_100k_UMAP_ann.h5ad"

# obs columns
EXPERIMENTAL_COLUMNS = [
    'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]',
    'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]',
    'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'o2_[%]',
    # 'days_of_culture'
]
SCATTER_COLUMNS = [
    'FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width'
]
LOG1P_EXP_COL = [
    "um171_[nm]", "um729_[µm]", "scf_[ng_ml]", 
    "butyzamide_[nm]", "o2_[%]",
    "sr1_[nm]", "mtg_[µm]", "rhflt3l_[ng_ml]", "gm-csf_[ng_ml]"
]
LOG21P_EXP_COL = [
    "ldl_[ng_ml]", "il3_[ng_ml]", "retinoic_acid_[µm]",
]

# adata keys
X_PCA_OBSM_KEY = "X_pca"
X_CHANNEL_OBSM_KEY = "X_channel"
X_SCATTER_OBSM_KEY = "X_scatter"
CONCAT_FEATS_OBSM_KEY_SUFFIX = "scatter_concat"
STANDARDIZED_FEATS_KEY_SUFFIX = "standardized"
CONDITION_CONCAT_OBSM_KEY = "protocol_axes_concat"

# training
TRAIN_BATCH_SIZE = 1024
VAL_BATCH_SIZE = 64
LR = 1e-4
NUM_GRAD_STEPS = 100_000
GRAD_STEPS_LOG_INTERVAL = 10
NUM_GRAD_ACCUMULATIONS_STEPS=100

# model
FLOW_TYPE = "rectified"
COUPLING_TYPE = "independent"
GENERATE_FROM_NOISE = True

# vf
ENCODE_STATE = True
ENCODE_TIME = True
USE_SINUSOIDAL_TIME_FEATURES = True
ENCODE_CONDITIONS = True

STATE_LATENT_DIM = 16
TIME_LATENT_DIM = 4
CONDITION_LATENT_DIM = 4

STATE_ENCODER_HIDDEN_DIMS = (128, 128)
VF_DECODER_HIDDEN_DIMS = (128, 128)
CONDITION_ENCODER_HIDDEN_DIMS = (64, 64)
TIME_ENCODER_HIDDEN_DIMS = (64, 64)


# paths
if IS_RUN_LOCALLY:
    DEVICE = "mps"
else:
    DEVICE = "cuda"

# sampling
N_SAMPLES = 500

VAL_SIZE = 3e-1

# reproducibility
SEED = 42
set_reproducibility(SEED)


## Data.

### Reading data.

In [3]:
adata = sc.read_h5ad(H5AD_PATH)
adata

AnnData object with n_obs × n_vars = 100000 × 21
    obs: 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_number', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width', 'Time Stamp', 'batch', 'leiden', 'cell_type'
    var: 'n', 'channel', 'marker', '$PnD', '$PnB', '$PnR', '$PnG', '$PnE', 'signal_type', 'antibody'
    uns: 'cell_type_colors', 'leiden', 'leiden_colors', 'meta', 'neighbors', 'processing_steps', 'umap'
    obsm: 'X_pca', 'X_umap'
    layers: 'positives', 'raw'
    

### Annotating conditions.

In [4]:
# concatenated experimental variable
adata = get_log_transformed_experimental_variables(
    adata,
    EXPERIMENTAL_COLUMNS,
    LOG1P_EXP_COL,
    LOG21P_EXP_COL,
    CONDITION_CONCAT_OBSM_KEY
)
adata


AnnData object with n_obs × n_vars = 100000 × 21
    obs: 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_number', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width', 'Time Stamp', 'batch', 'leiden', 'cell_type'
    var: 'n', 'channel', 'marker', '$PnD', '$PnB', '$PnR', '$PnG', '$PnE', 'signal_type', 'antibody'
    uns: 'cell_type_colors', 'leiden', 'leiden_colors', 'meta', 'neighbors', 'processing_steps', 'umap'
    obsm: 'X_pca', 'X_umap', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm

### Splitting data.

In [5]:
train_adata, val_adata = train_test_split(adata, test_size=VAL_SIZE, random_state=SEED)
train_adata.shape, val_adata.shape

((70000, 21), (30000, 21))

### Normalizing scatter data and storing it in `.obsm`.

In [6]:
# retrieving scatter features and optionally normalizing them
X_scatter_train = train_adata.obs[SCATTER_COLUMNS].values
X_scatter_val =  val_adata.obs[SCATTER_COLUMNS].values
X_scatter_mean = X_scatter_train.mean(0)
X_scatter_std = X_scatter_train.std(0)
print("Standardizing scatter features")
X_scatter_train_std = (X_scatter_train - X_scatter_mean)/X_scatter_std
X_scatter_val_std = (X_scatter_val - X_scatter_mean)/X_scatter_std
assert np.isfinite(X_scatter_train_std).all()
assert np.isfinite(X_scatter_val_std).all()
train_adata.obsm[X_SCATTER_OBSM_KEY] = X_scatter_train
val_adata.obsm[X_SCATTER_OBSM_KEY] = X_scatter_val
train_adata.obsm[f"{X_SCATTER_OBSM_KEY}_{STANDARDIZED_FEATS_KEY_SUFFIX}"] = X_scatter_train_std
val_adata.obsm[f"{X_SCATTER_OBSM_KEY}_{STANDARDIZED_FEATS_KEY_SUFFIX}"] = X_scatter_val_std


Standardizing scatter features


/tmp/ipykernel_2591592/2104843479.py:11: ImplicitModificationWarning: Setting element `.obsm['X_scatter']` of view, initializing view as actual.
  train_adata.obsm[X_SCATTER_OBSM_KEY] = X_scatter_train
/tmp/ipykernel_2591592/2104843479.py:12: ImplicitModificationWarning: Setting element `.obsm['X_scatter']` of view, initializing view as actual.
  val_adata.obsm[X_SCATTER_OBSM_KEY] = X_scatter_val


### Normalizing channel data and storing it in `.obsm`.

In [7]:
# concatenating with channel features
train_adata.obsm[X_CHANNEL_OBSM_KEY] = train_adata.X
val_adata.obsm[X_CHANNEL_OBSM_KEY] = val_adata.X
X_channel_train = train_adata.X
X_channel_val = val_adata.X
X_channel_mean = X_channel_train.mean(0)
X_channel_std = X_channel_train.std(0)
print("Standardizing channel features")
X_channel_train_std = (X_channel_train - X_channel_mean)/X_channel_std
X_channel_val_std = (X_channel_val - X_channel_mean)/X_channel_std
assert np.isfinite(X_channel_train).all()
assert np.isfinite(X_channel_val).all()
train_adata.obsm[f"{X_CHANNEL_OBSM_KEY}_{STANDARDIZED_FEATS_KEY_SUFFIX}"] = X_channel_train_std
val_adata.obsm[f"{X_CHANNEL_OBSM_KEY}_{STANDARDIZED_FEATS_KEY_SUFFIX}"] = X_channel_val_std
train_adata, val_adata

Standardizing channel features


(AnnData object with n_obs × n_vars = 70000 × 21
     obs: 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_number', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width', 'Time Stamp', 'batch', 'leiden', 'cell_type'
     var: 'n', 'channel', 'marker', '$PnD', '$PnB', '$PnR', '$PnG', '$PnE', 'signal_type', 'antibody'
     uns: 'cell_type_colors', 'leiden', 'leiden_colors', 'meta', 'neighbors', 'processing_steps', 'umap'
     obsm: 'X_pca', 'X_umap', 'mtg_[µm]', 'rhflt3l_[ng_ml]',

### Concatenating scatter and channel features and storing it in `.obsm`.

In [8]:
train_adata.obsm[f"{X_CHANNEL_OBSM_KEY}_{STANDARDIZED_FEATS_KEY_SUFFIX}_{CONCAT_FEATS_OBSM_KEY_SUFFIX}"] = np.concatenate((X_channel_train_std, X_scatter_train_std), axis=1)
val_adata.obsm[f"{X_CHANNEL_OBSM_KEY}_{STANDARDIZED_FEATS_KEY_SUFFIX}_{CONCAT_FEATS_OBSM_KEY_SUFFIX}"] = np.concatenate((X_channel_val_std, X_scatter_val_std), axis=1)
train_adata.obsm[f"{X_CHANNEL_OBSM_KEY}_{CONCAT_FEATS_OBSM_KEY_SUFFIX}"] = np.concatenate((X_channel_train, X_scatter_train), axis=1)
val_adata.obsm[f"{X_CHANNEL_OBSM_KEY}_{CONCAT_FEATS_OBSM_KEY_SUFFIX}"] = np.concatenate((X_channel_val, X_scatter_val), axis=1)
train_adata, val_adata

(AnnData object with n_obs × n_vars = 70000 × 21
     obs: 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_number', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width', 'Time Stamp', 'batch', 'leiden', 'cell_type'
     var: 'n', 'channel', 'marker', '$PnD', '$PnB', '$PnR', '$PnG', '$PnE', 'signal_type', 'antibody'
     uns: 'cell_type_colors', 'leiden', 'leiden_colors', 'meta', 'neighbors', 'processing_steps', 'umap'
     obsm: 'X_pca', 'X_umap', 'mtg_[µm]', 'rhflt3l_[ng_ml]',

## Preparing Neural Network Configurations.

### Predefining mlp settings.

In [9]:
get_mlp_config = lambda hidden_dims, final_activation_class=torch.nn.Identity: {
    "hidden_dims": hidden_dims,
    "use_batchnorm": False,
    "use_dropout": False,
    "activation_class": torch.nn.ReLU,
    "final_activation_class": final_activation_class,
}
get_lm_config = lambda final_activation_class=torch.nn.Identity: get_mlp_config((), final_activation_class=final_activation_class)


### Predefining condition encoder settings.

In [10]:
get_cond_encoder_concat = lambda hidden_dims=CONDITION_ENCODER_HIDDEN_DIMS: {
    "perturbation_layers_before_pooling":{
        f"feats_{CONDITION_CONCAT_OBSM_KEY}_{CONDITION_CONCAT_OBSM_KEY}": 
        {
            "input_dim": adata.obsm[CONDITION_CONCAT_OBSM_KEY].shape[-1],
            "output_dim": CONDITION_LATENT_DIM,
            **get_mlp_config(
                hidden_dims
            )
        }
    },
    "perturbation_layers_after_pooling": get_lm_config()
}

## Training the generative model.

### Initializing `FlowMatching` object.

In [11]:
# initializing model and preparing data
model = FlowMatching(
    flow_type=FLOW_TYPE,
    coupling_type=COUPLING_TYPE,
    generate_from_noise=GENERATE_FROM_NOISE,
    device_id=DEVICE,
)


### Preparing data.

In [12]:
repr = X_PCA_OBSM_KEY
conditioning_settings = {
    "perturbations": (CONDITION_CONCAT_OBSM_KEY),
    "perturbations_in_obsm": (CONDITION_CONCAT_OBSM_KEY),
    "perturbation_reps": {CONDITION_CONCAT_OBSM_KEY: CONDITION_CONCAT_OBSM_KEY},
}
model.prepare_train_data(
    train_adata,
    repr,
    **conditioning_settings
)

### Defining VF configurations.

In [13]:
cond_settings = get_cond_encoder_concat()
vf_config = NeuralVelocityFieldConfig(
    model.train_data.state_data.shape[-1],
    encode_state=ENCODE_STATE,
    state_encoder_output_dim=STATE_LATENT_DIM,
    state_encoder_mlp_kwargs=get_mlp_config(STATE_ENCODER_HIDDEN_DIMS),
    encode_time=ENCODE_TIME,
    use_sinusoidal_time_features=USE_SINUSOIDAL_TIME_FEATURES,
    time_encoder_output_dim=TIME_LATENT_DIM,
    time_encoder_mlp_kwargs=get_mlp_config(TIME_ENCODER_HIDDEN_DIMS),
    use_guidance=len(conditioning_settings) > 0,
    encode_conditions=ENCODE_CONDITIONS,
    perturbation_encoder_output_dim=CONDITION_LATENT_DIM,
    perturbation_layers_before_pooling=cond_settings.get("perturbation_layers_before_pooling", None),
    perturbation_layers_after_pooling=cond_settings.get("perturbation_layers_after_pooling", None),
    decoder_mlp_kwargs=get_mlp_config(VF_DECODER_HIDDEN_DIMS),
    conditioning_type="resnet",
)
print(vf_config)

NeuralVelocityFieldConfig(flow_dim=20, encode_state=True, state_encoder_output_dim=16, state_encoder_mlp_kwargs={'hidden_dims': (128, 128), 'use_batchnorm': False, 'use_dropout': False, 'activation_class': <class 'torch.nn.modules.activation.ReLU'>, 'final_activation_class': <class 'torch.nn.modules.linear.Identity'>}, encode_time=True, use_sinusoidal_time_features=True, time_features_num_freqs=128, time_features_max_periods=10000, time_encoder_output_dim=4, time_encoder_mlp_kwargs={'hidden_dims': (64, 64), 'use_batchnorm': False, 'use_dropout': False, 'activation_class': <class 'torch.nn.modules.activation.ReLU'>, 'final_activation_class': <class 'torch.nn.modules.linear.Identity'>}, use_guidance=True, encode_conditions=True, perturbation_encoder_output_dim=4, perturbation_layers_before_pooling={'feats_protocol_axes_concat_protocol_axes_concat': {'input_dim': 13, 'output_dim': 4, 'hidden_dims': (64, 64), 'use_batchnorm': False, 'use_dropout': False, 'activation_class': <class 'torch.n

### Preparing neural velocity field.

In [14]:
print(f"Preparing model")
optimizer_class = torch.optim.Adam
model.prepare_model(
    vf_config,
    optimizer_class=optimizer_class,
    optimizer_kwargs={"lr": LR}
)
model.velocity_field


Preparing model


NeuralVelocityField(
  (vf_modules): ModuleDict(
    (x_encoder): MLPBlock(
      (net): Sequential(
        (0): Sequential(
          (0): Linear(in_features=20, out_features=128, bias=True)
          (1): ReLU()
        )
        (1): Sequential(
          (0): Linear(in_features=128, out_features=128, bias=True)
          (1): ReLU()
        )
        (2): Sequential(
          (0): Linear(in_features=128, out_features=16, bias=True)
          (1): Identity()
        )
      )
    )
    (time_encoder): MLPBlock(
      (net): Sequential(
        (0): Sequential(
          (0): Linear(in_features=128, out_features=64, bias=True)
          (1): ReLU()
        )
        (1): Sequential(
          (0): Linear(in_features=64, out_features=64, bias=True)
          (1): ReLU()
        )
        (2): Sequential(
          (0): Linear(in_features=64, out_features=4, bias=True)
          (1): Identity()
        )
      )
    )
    (condition_encoder): ConditionEncoder(
      (modules_dict): M

### Fitting the model.

In [ ]:
model.train(
    num_training_steps=NUM_GRAD_STEPS,
    train_batch_size=TRAIN_BATCH_SIZE,
    grad_steps_log_interval=GRAD_STEPS_LOG_INTERVAL,
    num_grad_accumulation_steps=NUM_GRAD_ACCUMULATIONS_STEPS,
)
model.trainer.plot_training_logs()

Loss: 1.5501:  57%|█████▋    | 57059/100000 [06:43<04:57, 144.49it/s]

## Inference.

### Preparing validation data.

In [ ]:
model.prepare_validation_data(val_adata)
perturbation_repr = model.train_data.data.perturbation_covariates[0]
X_validation = model.validation_data.state_data
E_validation = model.validation_data.perturbation_data[perturbation_repr]
n_channels = X_validation.shape[1]
e_unique = np.unique(E_validation, axis=0)

### Generating samples under each protocol.

In [ ]:
pred_data = {}

pbar = tqdm(range(e_unique.shape[0]))
for idx, e_val in enumerate(e_unique):
    # progress bat update
    pbar.set_description(f"Generating cells for protocol {idx}")
    pbar.update()

    # retrieving true cells
    idx_cells = (E_validation == e_val).all(-1)
    X_true = X_validation[idx_cells]

    # generating cells
    e_val = torch.from_numpy(e_val).float().to(DEVICE).unsqueeze(0)
    X1_hat = model.predict(
        {
            DataFields.PERTURBATION_DATA: {
                perturbation_repr: e_val,
            },
        },
        return_trajectory=False,
        num_samples=N_SAMPLES,
    ).detach().cpu().squeeze().numpy()
    pred_data[idx] = {
        PredictionFields.PREDICTION_DATA: X1_hat,
        DataFields.TARGET_STATE: X_true,
    }

## Validation.

### Computing metrics  GvR.


In [ ]:
compute_r2g_metrics = False
if compute_r2g_metrics:
    metrics_real2gen = {}
    pbar = tqdm(range(len(pred_data)))

    for idx, idx_dict in pred_data.items():
        # progress bar update
        pbar.set_description(f"Evaluating cells for protocol {idx}")
        pbar.update()

        # retrieve data
        x_gen = idx_dict[PredictionFields.PREDICTION_DATA]
        x_real = idx_dict[DataFields.TARGET_STATE]

        # compute metrics
        e_dist = compute_e_distance(x_gen, x_real)
        wsrt_dist = compute_wasserstein_distance(x_gen, x_real)
        mmd = compute_mmd(x_gen, x_real)
        r_squared = compute_r_squared(x_gen, x_real)


        # storing results
        metrics_real2gen[idx] = {
            "protocol_id": idx,
            "e_dist": e_dist,
            "wsrt_dist": wsrt_dist,
            "mmd": mmd,
            "r_squared": r_squared,
            "n_cells": x_real.shape[0],
        }

    df_metrics_r2g = pd.DataFrame(metrics_real2gen)
    df_metrics_r2g

### Heatmap of metrics GvR.

In [ ]:
compute_r2g_metrics = False
if compute_r2g_metrics:
    # Remove 'protocol_id' column and set protocol IDs as index
    df_heatmap = df_metrics_r2g.T.set_index('protocol_id')

    # Plot one heatmap per metric
    for metric in df_heatmap.columns:
        plt.figure(figsize=(12, 2))  # adjust height for readability
        sns.heatmap(
            df_heatmap[[metric]].T,  # select metric and transpose for 1 row
            cbar_kws={'label': metric}
        )
        plt.title(f"{metric} per Protocol")
        plt.xlabel("Protocol ID")
        plt.show()


### Computing metrics  GvG.

In [ ]:
compute_g2g_metrics = False
if compute_g2g_metrics:
    metrics_gen2gen = {}
    pbar = tqdm(range(len(pred_data)))

    for idx0, idx0_dict in pred_data.items():
        # progress bar update
        pbar.set_description(f"Evaluating cells for protocol {idx0}")
        pbar.update()

        # retrieve data
        x0_gen = idx0_dict[PredictionFields.PREDICTION_DATA]

        for idx1, idx1_dict in pred_data.items():
            if idx1 <= idx0:
                continue

            # retrieve data
            x1_gen = idx1_dict[PredictionFields.PREDICTION_DATA]

            # generated: compute metrics
            e_dist_gen = compute_e_distance(x0_gen, x1_gen)
            # wsrt_dist_gen = compute_wasserstein_distance(x0_gen, x1_gen)
            mmd_gen = compute_mmd(x0_gen, x1_gen)
            r_squared_gen = compute_r_squared(x0_gen, x1_gen)
            # storing results
            metrics_gen2gen[idx] = {
                "protocol_id0": idx0,
                "protocol_id1": idx1,
                "e_dist": e_dist,
                # "wsrt_dist": wsrt_dist,
                "mmd": mmd,
                "r_squared": r_squared,
            }

    df_metrics_g2g = pd.DataFrame(metrics_gen2gen)


### Computing metrics  RvR.


In [ ]:
compute_r2r_metrics = False
if compute_r2r_metrics:
    metrics_real2real = {}
    pbar = tqdm(range(len(pred_data)))

    for idx0, idx0_dict in pred_data.items():
        # progress bar update
        pbar.set_description(f"Evaluating cells for protocol {idx0}")
        pbar.update()

        # retrieve data
        x0_real = idx0_dict[DataFields.TARGET_STATE]

        for idx1, idx1_dict in pred_data.items():
            if idx1 <= idx0:
                continue

            # retrieve data
            x1_real = idx1_dict[DataFields.TARGET_STATE]

            # real: compute metrics
            e_dist_gen = compute_e_distance(x0_real, x1_real)
            # wsrt_dist_gen = compute_wasserstein_distance(x0_real, x1_real)
            mmd_gen = compute_mmd(x0_real, x1_real)
            r_squared_gen = compute_r_squared(x0_real, x1_real)
            # storing results
            metrics_real2real[idx] = {
                "protocol_id0": idx0,
                "protocol_id1": idx1,
                "e_dist": e_dist,
                # "wsrt_dist": wsrt_dist,
                "mmd": mmd,
                "r_squared": r_squared,
            }

    df_metrics_r2r = pd.DataFrame(metrics_real2real)


### KDE plot for each protocol.

In [ ]:
plot_per_protocol_kde = True
if plot_per_protocol_kde:
    for protocol_idx, protocol_data in enumerate(e_unique):
        idx_cells = (E_validation == protocol_data).all(-1)
        X_subset = model.validation_data.state_data[idx_cells, :]
        X_gen = pred_data[protocol_idx][PredictionFields.PREDICTION_DATA]

        fig, ax = plt.subplots(1, n_channels, figsize=(45, 5))
        fig.suptitle(f"KDE plot of channel features: {protocol_idx}", fontsize=16)
        for i in range(n_channels):
            sns.kdeplot(X_subset[:, i], ax=ax[i], color="green", label="Real")
            sns.kdeplot(X_gen[:, i], ax=ax[i], color="red", label="Generated")
            ax[i].grid(True)
            ax[i].set_title(f"PC{i}")

        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.show()